# Notebook IV — Soil Constraints
<hr>

## Overview
This notebook runs **Module IV** — it applies soil yield reduction factors (fc4) to the climate-adjusted yield maps from NB3. Soil constraints account for edaphic (soil-related) limitations: seven soil quality indicators are combined into a single soil rating per Soil Mapping Unit (SMU), then applied pixel-by-pixel.

### Prerequisites
NB1, NB2, and NB3 must have been run first. This notebook reads:
- NB3 outputs: `data_output/NB3/clim_maiz_yield_rain.tif`, `clim_maiz_yield_irr.tif`
- `data_input/PK_Soil.tif` — HWSD v2.0 soil map (87×108, must match pipeline grid)
- Excel files: `maiz_soil_params_rain/irr.xlsx`, `maiz_soil_characteristics_topsoil/subsoil.xlsx`

> **`PK_Soil.tif` must be generated before running this notebook.**
> Run `scripts/download_rasters_pakistan.py` after placing `HWSD2.bil` in `data_input/raster_raw/`.

### What this notebook produces (saved to `data_output/NB4/`)
| Output file | Description |
|---|---|
| `soil_clim_adj_yield_maiz_rain.tif` | fc4-adjusted rainfed yield (kg/ha) |
| `soil_clim_adj_yield_maiz_irr.tif` | fc4-adjusted irrigated yield (kg/ha) |
| `fc4_maiz_rain.tif` | Soil reduction factor — rainfed (0–1) |
| `fc4_maiz_irr.tif` | Soil reduction factor — irrigated (0–1) |

Prepared by Geoinformatics Center, AIT — Pakistan adaptation by Saqib Ali
<hr>

### Step 1 — Connect to Google Drive (Colab only)

Mount Google Drive to access the repo at `/content/drive/MyDrive/PK-PyAEZ/`.

> **Skip** if running locally.

In [ ]:
# ── Google Colab setup ──────────────────────────────────────────────
# Run these steps ONLY when using Google Colab.

# Step 1: Clone the repository (first time only)
# !git clone https://github.com/YOUR_USERNAME/PK-PyAEZ.git /content/PK-PyAEZ

# Step 2 (optional): Mount Google Drive to persist outputs across sessions
# from google.colab import drive
# drive.mount('/content/drive')

### Step 2 — Install dependencies (Colab only)

Installs GDAL, Numba, and all required packages, then installs `pyaez` in editable mode from the Drive copy.

> **Skip** if running locally with the conda environment already set up.

In [ ]:
import sys

if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run(['apt-get', 'install', '-y', '-q', 'gdal-bin', 'libgdal-dev', 'python3-gdal'],
                   capture_output=True)
    _ver = subprocess.check_output(['gdal-config', '--version']).decode().strip()
    subprocess.run(['pip', 'install', '-q',
                    f'GDAL=={_ver}', 'numba', 'openpyxl', 'rasterio', 'requests', 'scipy', 'pandas'],
                   capture_output=True)
    subprocess.run(['pip', 'install', '-q', '-e', '/content/drive/MyDrive/PK-PyAEZ'], capture_output=True)
    print(f"Colab setup complete — GDAL {_ver}")

### Step 3 — Import Python libraries

| Library | Purpose |
|---|---|
| `numpy` | Array operations on yield and soil maps |
| `matplotlib` | Plots before/after yield maps and fc4 factor |
| `pandas` | Reads soil Excel parameter sheets |
| `gdal` (osgeo) | Opens input GeoTIFFs; saves output rasters |
| `os`, `sys` | File path and module path management |

`gdal.UseExceptions()` silences the GDAL 4.0 FutureWarning.

In [ ]:
'''import supporting libraries'''
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
try:
    from osgeo import gdal
except:
    import gdal
gdal.UseExceptions()
import sys

### Step 4 — Set working directory

Sets CWD to the repo root so all `./data_input/` and `./data_output/` paths resolve correctly.

- **Colab**: `/content/drive/MyDrive/PK-PyAEZ`
- **Local**: `..` (one level up from `tutorials/`)

In [ ]:
'Set the working directory'
import sys, os

if 'google.colab' in sys.modules:
    work_dir = '/content/drive/MyDrive/PK-PyAEZ'
else:
    work_dir = '..'

os.chdir(work_dir)
sys.path.insert(0, os.getcwd())
os.getcwd()

### Step 5 — Create output folder

Creates `data_output/NB4/` if it does not already exist. All fc4-adjusted yield maps and reduction factor rasters are saved there.

In [ ]:
import os
folder_path = './data_output/NB4/'
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print("Folder created successfully.")
else:
    print("Folder already exists.")


<hr>

## MODULE IV — Soil Constraints

### Step 6 — Initialise SoilConstraints and load input data

Creates the `soil_constraints` object and loads three inputs:

| Variable | Source | Description |
|---|---|---|
| `yield_map_rain` | `NB3/clim_maiz_yield_rain.tif` | fc3-adjusted rainfed yield (kg/ha) |
| `yield_map_irr` | `NB3/clim_maiz_yield_irr.tif` | fc3-adjusted irrigated yield (kg/ha) |
| `soil_map` | `PK_Soil.tif` | HWSD v2.0 Soil Mapping Unit (SMU) codes — must be 87×108 |

> **Shape check:** all three arrays must be 87×108. If `soil_map` is 58×72, regenerate `PK_Soil.tif` using `scripts/download_rasters_pakistan.py`.

In [ ]:
'''importing library'''

from pyaez import SoilConstraints
soil_constraints = SoilConstraints.SoilConstraints()

from pyaez import UtilitiesCalc
obj_utilities = UtilitiesCalc.UtilitiesCalc()

In [ ]:
'''reading data'''
basefilepath = './data_input/PK_Admin.tif'

yield_map_rain = gdal.Open('./data_output/NB3/clim_maiz_yield_rain.tif').ReadAsArray()
yield_map_irr = gdal.Open('./data_output/NB3/clim_maiz_yield_irr.tif').ReadAsArray()

soil_map = gdal.Open('./data_input/PK_Soil.tif').ReadAsArray()

### Step 7 — Load soil suitability reduction factors from Excel

Reads crop-specific soil suitability ratings from two Excel sheets — one for rainfed, one for irrigated. Each sheet maps soil quality combinations to an fc4 reduction factor (0–1).

| File | Water supply | Description |
|---|---|---|
| `maiz_soil_params_rain.xlsx` | Rainfed | fc4 LUT for each soil quality class — rainfed |
| `maiz_soil_params_irr.xlsx` | Irrigated | fc4 LUT for each soil quality class — irrigated |

To adapt for a different crop, replace these files with the equivalent crop-specific Excel sheets.

In [ ]:
soil_constraints.importSoilReductionSheet(rain_sheet_path='./data_input/maiz_soil_params_rain.xlsx',
                             irr_sheet_path='./data_input/maiz_soil_params_irr.xlsx')

In [ ]:
'''calculate soil qualities and ratings and applying soil constraints - Rain-fed'''

soil_constraints.calculateSoilQualities(irr_or_rain='R', topsoil_path= './data_input/maiz_soil_characteristics_topsoil.xlsx',
                                         subsoil_path= './data_input/maiz_soil_characteristics_subsoil.xlsx') # I: Irrigated, R: Rain-fed

# Calculate Soil rating for each SMU at a given input/management level
soil_constraints.calculateSoilRatings('H') # L: Low, I: Intermediate, H: High input

### Step 8 — Calculate and review soil qualities (Rainfed)

`calculateSoilQualities(irr_or_rain='R', ...)` reads topsoil and subsoil Excel sheets and computes **7 soil quality indicators** for each SMU:

| # | Soil Quality | What it measures |
|---|---|---|
| SQ1 | Nutrient availability | N, P, K, organic carbon |
| SQ2 | Nutrient retention | CEC, base saturation |
| SQ3 | Rooting conditions | Depth, gravel, texture |
| SQ4 | Oxygen availability | Drainage class |
| SQ5 | Excess salts | Salinity, sodicity |
| SQ6 | Toxicities | Al, Fe, Mn saturation |
| SQ7 | Workability | Texture, gravel for tillage |

`calculateSoilRatings('H')` combines the 7 qualities into one fc4 value per SMU.

| Input level | Code | Description |
|---|---|---|
| High | `'H'` | Mechanised farming, full inputs |
| Intermediate | `'I'` | Partial mechanisation |
| Low | `'L'` | Subsistence / hand tools |

The table below shows the computed quality scores — inspect to verify soil data loaded correctly.

In [ ]:
np.set_printoptions(precision = 2)
# result as pandas array
soil_constraints.getSoilQualities()

### Step 9 — Review soil ratings per SMU (Rainfed)

`getSoilRatings()` returns a table where:
- **Column 1**: SMU code (matching values in `PK_Soil.tif`)
- **Column 2**: fc4 value (0–1) — the combined soil suitability reduction factor

Higher fc4 = better soil for this crop and management level. This lookup table is then spatially applied to each pixel via the SMU map.

In [ ]:
soil_constraints.calculateSoilRatings(input_level= 'H')

# 1st column: SMUs, 2nd column, soil constraint factors
print(soil_constraints.getSoilRatings())

In [ ]:
# getting the soil ratings
soil_constraints.getSoilRatings()

### Step 10 — Apply soil constraints (Rainfed)

`applySoilConstraints(soil_map, yield_map_rain)` assigns fc4 to each pixel by looking up its SMU code, then multiplies the yield by fc4:

```
yield_adjusted = yield_input × fc4(SMU)
```

`getSoilSuitabilityMap()` returns the spatial fc4 map for visualisation and saving.

- **Output:** `yield_map_rain_m4` (kg/ha), `fc4_rain` (0–1 per pixel)
- **Saved to:** `data_output/NB4/soil_clim_adj_yield_maiz_rain.tif`, `fc4_maiz_rain.tif`

In [ ]:
yield_map_rain_m4 = soil_constraints.applySoilConstraints(soil_map, yield_map_rain)

## get classified output
# yield_map_rain_class_m4 = obj_utilities.classifyFinalYield(yield_map_rain_m4)

fc4_rain = soil_constraints.getSoilSuitabilityMap()

In [ ]:
'''visualize results'''
plt.figure(1, figsize=(25,9))
plt.subplot(1,3,1)
plt.imshow(yield_map_rain, vmax = np.max([yield_map_rain_m4, yield_map_rain]))
plt.colorbar(shrink=0.8)
plt.title('Original Rainfed Yield Maize')

plt.subplot(1,3,2)
plt.imshow(yield_map_rain_m4, vmax = np.max([yield_map_rain_m4, yield_map_rain]))
plt.colorbar(shrink=0.8)
plt.title('Soil Constrainted Rainfed Yield Maize')

plt.subplot(1,3,3)
plt.imshow(fc4_rain,)
plt.colorbar(shrink=0.8)
plt.title('Soil Constraint Factor (fc4) Rainfed')


In [ ]:
# ''save result'''

obj_utilities.saveRaster(basefilepath, './data_output/NB4/soil_clim_adj_yield_maiz_rain.tif', yield_map_rain_m4)
obj_utilities.saveRaster(basefilepath, './data_output/NB4/fc4_maiz_rain.tif', fc4_rain)

---
### Step 11 — Apply soil constraints (Irrigated)

Same workflow as Steps 8–10, but using `irr_or_rain='I'` and the irrigated fc4 ratings. Irrigation removes moisture stress, so soil workability and drainage have a larger relative weight than for rainfed.

- **Output:** `yield_map_irr_m4` (kg/ha), `fc4_irr` (0–1 per pixel)
- **Saved to:** `data_output/NB4/soil_clim_adj_yield_maiz_irr.tif`, `fc4_maiz_irr.tif`

In [ ]:
'''calculate soil qualities and ratings and applying soil constraints - Irrigated'''

soil_constraints.calculateSoilQualities(irr_or_rain='I', topsoil_path= './data_input/maiz_soil_characteristics_topsoil.xlsx',
                                         subsoil_path= './data_input/maiz_soil_characteristics_subsoil.xlsx') # I: Irrigated, R: Rain-fed

# Calculate Soil rating for each SMU at a given input/management level
soil_constraints.calculateSoilRatings('H') # L: Low, I: Intermediate, H: High input

#### Review soil qualities (Irrigated)

Same 7 soil quality indicators recalculated for irrigated conditions — drainage (SQ4) and workability (SQ7) receive adjusted weights.

In [ ]:
np.set_printoptions(precision = 2)
# result as pandas array
soil_constraints.getSoilQualities()

#### Review soil ratings per SMU (Irrigated)

fc4 values per SMU under irrigated management. Compare with the rainfed table above — differences indicate where drainage or workability is limiting rainfed more than irrigated performance.

In [ ]:
# 1st column: SMUs, 2nd column, soil constraint factors
print(soil_constraints.getSoilRatings())

#### Apply irrigated soil constraints and extract results

In [ ]:
yield_map_irr_m4 = soil_constraints.applySoilConstraints(soil_map, yield_map_irr)

## get classified output
# yield_map_rain_class_m4 = obj_utilities.classifyFinalYield(yield_map_rain_m4)

fc4_irr = soil_constraints.getSoilSuitabilityMap()

In [ ]:
'''visualize results'''
plt.figure(1, figsize=(25,9))
plt.subplot(1,3,1)
plt.imshow(yield_map_irr, vmax = np.max([yield_map_irr_m4, yield_map_irr]))
plt.colorbar(shrink=0.8)
plt.title('Original Irrigated Yield Maize')

plt.subplot(1,3,2)
plt.imshow(yield_map_irr_m4, vmax = np.max([yield_map_irr_m4, yield_map_irr]))
plt.colorbar(shrink=0.8)
plt.title('Soil Constrainted Irrigated Yield Maize')

plt.subplot(1,3,3)
plt.imshow(fc4_rain, vmax = 1, vmin = 0)
plt.colorbar(shrink=0.8)
plt.title('Soil Constraint Factor (fc4) Irrigated')

In [ ]:
# saving the outputs
obj_utilities.saveRaster(basefilepath, './data_output/NB4/soil_clim_adj_yield_maiz_irr.tif', yield_map_irr_m4)
obj_utilities.saveRaster(basefilepath, './data_output/NB4/fc4_maiz_irr.tif',fc4_irr)

<hr>

### END OF MODULE IV — Soil Constraints

**Next step → NB5:** Apply terrain constraints (fc5) to the yields adjusted here.

Inputs passed forward to NB5:
- `data_output/NB4/soil_clim_adj_yield_maiz_rain.tif`
- `data_output/NB4/soil_clim_adj_yield_maiz_irr.tif`

<hr>